# GymRAG — Supplement Catalog Ingestion

Turns the supplement guide PDFs in `../data/` into a searchable Qdrant collection (`gymrag-supplements-openai-qdrant`) that the GymRAG advisor answers from. Run it top to bottom once, and again whenever the PDFs change.

**LLM:** `gpt-4o-mini` (OpenAI) · **Embedding:** `text-embedding-3-small` (OpenAI) · **Vector store:** Qdrant (hybrid dense + sparse)

## 1. Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from ragwire import RAGWire, setup_logging
import ragwire

logger = setup_logging(log_level="INFO")
print(ragwire.__version__)

In [ ]:
# force_recreate: true in rag_pipeline.yaml -> this drops and recreates the collection.
rag = RAGWire("rag_pipeline.yaml")

## 2. Prepare the catalog: one clean file per product

Each guide bundles 20 products into one PDF, and PDF text extraction scrambles the spec tables: every label is printed first and the values follow later in the same order, mixed with page footers. RAGWire's metadata extractor also runs once per **file**, so ingesting a whole guide would stamp the first product's metadata onto every chunk.

This cell fixes both problems deterministically, before any LLM sees the text:

- **Split** each guide on its numbered product headings (`07. Brand - Product`).
- **Clean** each block by dropping page footers and re-joining wrapped lines.
- **Repair** the spec table by pairing known labels with values in order, giving `Label: value` lines.
- **Stage** each product as its own small Markdown file with its category stated explicitly.

The extractor then only has to copy clean `Label: value` lines, which is what keeps the catalog accurate.

In [ ]:
import re
import shutil
from pathlib import Path

DATA_DIR = Path("../data")
STAGE_DIR = DATA_DIR / "_staged_products"

# One guide per category. Add a row here to ingest another guide.
CATALOGS = {
    "whey_protein": DATA_DIR / "Whey_Protein_Guide.pdf",
    "creatine": DATA_DIR / "Creatine_Ultimate_Guide.pdf",
    "pre_workout": DATA_DIR / "PreWorkout_Ultimate_Guide.pdf",
    "post_workout": DATA_DIR / "PostWorkout_Recovery_Guide.pdf",
    "multivitamin": DATA_DIR / "Multivitamin_Ultimate_Guide.pdf",
}

# Every label that can appear in a product's spec table, in any guide.
COMMON_LABELS = ["Quantity / Servings", "Price"]
CATEGORY_LABELS = {
    "whey_protein": ["Protein per Serving", "Protein Source"],
    "creatine": ["Creatine per Serving", "Creatine Form"],
    "pre_workout": ["Caffeine per Serving", "L-Citrulline (Pump)", "Beta-Alanine (Endurance)"],
    "post_workout": ["Amino Acids per Serving", "Glutamine per Serving", "Electrolytes"],
    "multivitamin": ["Serving Size", "Key Nutrients", "Best For"],
}
REVIEW_LABEL = "Review & Efficacy"

HEADING_RE = re.compile(r"^[ \t\f]*(\d{1,3})\.\s+(.+?)\s+-\s+(.+)$", re.MULTILINE)
FOOTER_RE = re.compile(r"^(Page \d+|.+ - Sample Edition \d{4})$")


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")[:60]


def paragraphs(block: str) -> list[str]:
    """Blank-line separated paragraphs, wrapped lines re-joined, footers dropped."""
    out = []
    for para in re.split(r"\n\s*\n", block):
        text = " ".join(line.strip() for line in para.splitlines() if line.strip())
        if text and not FOOTER_RE.match(text):
            out.append(text)
    return out


def split_guide(text: str, category: str) -> list[dict]:
    labels = COMMON_LABELS + CATEGORY_LABELS[category] + [REVIEW_LABEL]
    headings = list(HEADING_RE.finditer(text))
    products = []
    for i, h in enumerate(headings):
        end = headings[i + 1].start() if i + 1 < len(headings) else len(text)
        body = paragraphs(text[h.end():end])

        overview = next((p for p in body if p.startswith("Brand Overview:")), "")
        rest = [p for p in body if p != overview]
        found_labels = [p for p in rest if p in labels]
        values = [p for p in rest if p not in labels]
        if found_labels != labels or len(values) != len(labels):
            raise ValueError(f"{category} #{h.group(1)}: expected {labels}, got labels={found_labels} values={values}")

        products.append({
            "number": int(h.group(1)),
            "brand": h.group(2).strip(),
            "name": h.group(3).strip(),
            "overview": overview.removeprefix("Brand Overview:").strip(),
            "specs": dict(zip(labels, values)),
        })
    return products


def to_markdown(category: str, p: dict) -> str:
    lines = [
        f"# {p['brand']} - {p['name']}",
        f"Category: {category}",
        f"Brand / Make: {p['brand']}",
        f"Product Name: {p['name']}",
        f"Brand Overview: {p['overview']}",
    ]
    lines += [f"{label}: {value}" for label, value in p["specs"].items()]
    return "\n".join(lines) + "\n"


if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True)

staged = []
for category, path in CATALOGS.items():
    result = rag.loader.load(str(path))
    if not result["success"]:
        raise RuntimeError(f"Failed to load {path}: {result['error']}")
    products = split_guide(result["text_content"], category)
    for p in products:
        out = STAGE_DIR / f"{category}_{p['number']:02d}_{slugify(p['brand'] + ' ' + p['name'])}.md"
        out.write_text(to_markdown(category, p), encoding="utf-8")
        staged.append(out)
    print(f"{category:<13} {len(products)} products staged")

longest = max(len(p.read_text(encoding="utf-8")) for p in staged)
print(f"\nStaged {len(staged)} single-product files in {STAGE_DIR}/ (longest: {longest} chars)")

In [ ]:
# Inspect one staged file: this is exactly what the metadata extractor reads.
print(sorted(STAGE_DIR.glob("pre_workout_02_*.md"))[0].read_text(encoding="utf-8"))

## 3. Ingest

Embeds every staged product (dense + sparse) and stores its extracted metadata in Qdrant. File-hash dedup makes re-runs cheap when `force_recreate` is `false`.

In [ ]:
stats = rag.ingest_directory(str(STAGE_DIR))
stats

## 4. Sanity-check retrieval

In [ ]:
def show(results):
    if not results:
        print("No results.")
        return
    for doc in results:
        m = doc.metadata
        print(f"- [{m.get('category')}] {m.get('brand_or_make')} - {m.get('product_name')} | {m.get('price')}")


# Plain semantic search, no category filter
show(rag.retrieve("I can't recover between sessions", top_k=3))

In [ ]:
# Category-filtered searches
show(rag.retrieve("strong pre-workout with high caffeine", top_k=3, filters={"category": "pre_workout"}))
show(rag.retrieve("whey isolate with 25g protein", top_k=3, filters={"category": "whey_protein"}))
show(rag.retrieve("vegan multivitamin", top_k=3, filters={"category": "multivitamin"}))

In [ ]:
# Every product should carry its own category-specific fields
values = rag.get_field_values(["category"], limit=50)
print(values)